# 300_HuggingFace_pipeline

파이프라인은 추론을 위해 모델을 사용하는 훌륭하고 쉬운 방법입니다.

이러한 파이프라인은 라이브러리에서 대부분의 복잡한 코드를 추상화하는 개체로, Named Entity Recognition, Masked Language Modeling, 감정 분석, Feature Extraction 및 Question Answering.을 비롯한 여러 task 전용의 간단한 API를 제공합니다.

In [3]:
!pip install -q transformers datasets
!pip install -q sentencepiece
!pip install -q kobert-transformers

In [4]:
from transformers import pipeline

라이브러리는 **텍스트의 감정 분석과 같은 자연어 이해(NLU)** 및 **새 텍스트로 프롬프트를 완성하거나 다른 언어로 번역하는 것과 같은 자연어 생성(NLG)** 작업을 위해 사전 훈련된 모델을 다운로드합니다.

먼저 pipeline API를 쉽게 활용하여 추론에서 **사전 훈련된 모델**을 빠르게 사용하는 방법을 살펴보겠습니다. 그런 다음, 라이브러리가 어떻게 이러한 모델에 대한 액세스를 제공하고 **데이터를 사전 처리**하는 데 도움이 되는지 확인 할 것입니다.

## pipeline 으로 작업 시작하기

- 주어진 task 에서 사전 훈련된 모델을 사용하는 가장 쉬운 방법은  `pipeline`을 사용하는 것입니다.

🤗 Transformers 라이브러리는 기본적으로 다음 task를 제공합니다.

- **기계 번역(Translation)**: 다른 언어로 된 텍스트를 번역합니다.  
- **감정 분석(Text Classification)**: 텍스트는 긍정적인가 부정적인가?
- **텍스트 생성(Text Generation)**: 프롬프트를 제공하면 모델이 다음을 생성합니다.
- **이름 개체 인식(NER)**: 입력 문장에서 각 단어를 나타내는 개체(사람, 장소, 등.)
- **질문 답변(Question Answering)**: 모델에 일부 컨텍스트와 질문을 제공하고 컨텍스트에서 답변을 추출합니다.
- **마스킹된 텍스트 채우기(Fill-Mask)**: 마스킹된 단어가 있는 텍스트(예: `[MASK]`로 대체)가 주어지면 공백을 채웁니다.
- **요약(Summarization)**: 긴 텍스트의 요약을 생성합니다.
- **특징 추출(Feature Extraction)**: 텍스트의 텐서 표현을 반환합니다.
- **Zero-Shot 분류(Zero-Shot Classification)**


### pretrained models : https://huggingface.co/models

## 기계 번역

- korean pretrained model : https://huggingface.co/Helsinki-NLP/opus-mt-ko-en  

- Helsinki-NLP : University of Helsinki 에서 작성한 다양한 언어 모델 그룹

In [5]:
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-ko-en")

translator("나는 너를 사랑해"), translator("언제나 당신에게 감사함을 느끼고 있습니다.")

RuntimeError: Failed to import transformers.models.marian.modeling_tf_marian because of the following error (look up to see its traceback):
No module named 'keras.engine'

## 감정 분석

In [ ]:
classifier = pipeline('sentiment-analysis')

- 이 명령을 처음 입력하면 사전 훈련된 모델과 해당 토크나이저가 다운로드되어 캐시됩니다.  

- 토크나이저는 모델의 텍스트를 사전 처리한 다음 예측을 담당하는 역할을 합니다.  

- 파이프라인은 모든 것을 함께 그룹화하고 예측을 읽을 수 있도록 사후 처리합니다. 예를 들어:

In [ ]:
classifier('We are very happy to show you the Transformers library.')

사전 처리된 다음 모델에 *batch*로 공급하면 다음과 같은 사전 목록을 반환합니다.

In [ ]:
results = classifier([
                      "We are very happy to show you the Transformers library.",
                       "We hope you don't hate it."
                        ])
for result in results:
    print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

## 한국어 감정분석

- NSMC(Naver Sentiment Movie Corpus) 로 미세 조정된 BERT 다국어 basecase 모델 : https://huggingface.co/sangrimlee/bert-base-multilingual-cased-nsmc

In [ ]:
classifier_ko = pipeline('sentiment-analysis',
                      model="sangrimlee/bert-base-multilingual-cased-nsmc")

In [ ]:
print(classifier_ko("오늘은 정말 즐거운 날이다. 행복하다."))
print(classifier_ko("기분이 꿀꿀해서 술이나 한잔 해야겠다."))

In [ ]:
classifier_ko([
               "언제나 당신에게 감사함을 느끼고 있습니다.",
               "너한테는 별로 좋은 기억이 없어."
               ])

- 자동 별점 부여

In [ ]:
model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
kor_classifier = pipeline('sentiment-analysis', model=model_name)

results = kor_classifier(["다시는 보고 싶지 않은 짜증나는 영화",
                              "아주 재미있는 영화",
                              "정말 재미없는 영화였다",
                              "이 영화 망할거야",
                              "이 영화 최고",
                              "보통 영화"])

for result in results:
    print(f"label: {result['label']}, with score: {round(result['score'], 4)}")

## Zero Shot Pipeline - 처음 보는 문장의 category 분류

### 제로샷 분류
- 라벨이 지정되지 않은 텍스트를 분류  
- 분류에 사용할 레이블을 지정할 수 있으므로 사전 훈련된 모델의 레이블에 의존할 필요가 없습니다.   

이 파이프라인을 사용하기 위해 데이터 모델을 미세 조정할 필요가 없기 때문에 제로샷이라고 합니다. 원하는 라벨 목록에 대한 확률 점수를 직접 반환할 수 있습니다.

In [ ]:
classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformmers library",
    candidate_labels=["education", "politics", "business"],
)

### 마스킹된 텍스트 채우기
- 주어진 텍스트의 빈칸을 채우는 것  

top_k 인수는 표시할 가능성의 수를 제어합니다. 여기서 모델은 종종 마스크 토큰이라고 하는 특수 <mask> 단어를 채웁니다. 다른 마스크 채우기 모델에는 다른 마스크 토큰이 있을 수 있으므로 다른 모델을 탐색할 때 항상 적절한 마스크 단어를 확인하는 것이 좋습니다.

In [ ]:
unmasker = pipeline("fill-mask")

# 주어진 문장에서 <mask>를 채우는 예측 단어를 찾음
result = unmasker("This course will teach you all about <mask> models.", top_k=2)

print(result)

### 명명된 엔터티 인식
명명된 엔터티 인식(NER)은 모델이 입력 텍스트의 어느 부분이 사람, 위치 또는 조직과 같은 엔터티에 해당하는지 찾아야 하는 작업입니다.

In [ ]:
ner = pipeline("ner", grouped_entities=True)
ner("My name is Sylvian and I work at Hugging Face in Brooklyn")

위에서 모델은 Sylvain이 사람(PER), Hugging Face가 조직(ORG), Brooklyn이 위치(LOC)임을 올바르게 식별했습니다.

파이프라인 생성 함수에 grouped_entities=True 옵션을 전달하여 동일한 엔터티에 해당하는 문장 부분을 함께 재그룹화하도록 파이프라인에 지시합니다. 여기서 모델은 "Hugging"과 "Face"를 단일 조직으로 올바르게 그룹화했습니다. 이름은 여러 단어로 구성됩니다. 실제로 전처리는 일부 단어를 더 작은 부분으로 분할하기도 합니다. 예를 들어 Sylvain은 S, ##yl, ##va, ##in의 네 부분으로 나뉩니다. 사후 처리 단계에서 파이프라인은 해당 조각을 성공적으로 재그룹화했습니다.

### 질문 답변
질문 답변 파이프라인은 주어진 컨텍스트의 정보를 사용하여 질문에 답변합니다.

In [ ]:
question_answerer = pipeline("question-answering")
question_answerer(
    question="Where do I work ?",
    context = "My name is Sylvian and I work at Hugging Face in Brooklyn"
)

### 요약
요약은 텍스트에서 참조된 모든 중요한 측면을 유지하면서 텍스트를 더 짧은 텍스트로 줄이는 작업입니다.

In [ ]:
from transformers import pipeline

# 문서 요약
summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)

## Text 생성
- 프롬프트를 제공하면 모델이 나머지 텍스트를 생성하여 프롬프트를 자동 완성  
- 텍스트 생성에는 무작위성이 포함되므로 아래와 같은 결과가 나오지 않는 것이 정상입니다.

In [ ]:
# "text-generation" 파이프라인을 로드
generator = pipeline("text-generation")

# 주어진 텍스트 다음에 자동으로 생성된 텍스트 생성
result = generator("In this course, we will teach you how to")

print(result)

### 파이프라인에서 허브의 모델 사용
- 허브에서 특정 모델을 선택하여 특정 작업(예: 텍스트 생성)을 위한 파이프라인에서 사용할 수 있습니다.
- Model Hub(https://huggingface.co/models)

- DistilGPT2  
DistilGPT2(Distilled-GPT2)는 GPT-2(Generative Pre-trained Transformer 2)의 사전 훈련된 가장 작은 영어 모델입니다. HuggingFace가 만들었습니다.  
- language 태그를 클릭하여 모델 검색을 구체화하고 다른 언어로 텍스트를 생성할 모델을 선택할 수 있습니다. 모델 허브에는 여러 언어를 지원하는 다국어 모델에 대한 체크포인트도 포함되어 있습니다.

In [ ]:
generator = pipeline("text-generation", model="distilgpt2")
generator(
        "In this course, we will teach you how to",
        max_length=30,
        num_return_sequences=2)

### 한글 Text 생성

- SKT 에서 2021년 5월에 만든 `skt/ko-gpt-trinity-1.2B-v0.5` model 사용 (https://huggingface.co/skt/ko-gpt-trinity-1.2B-v0.5)  
- (참고) download 시간 오래 걸림 - 4.8GB

In [ ]:
generator = pipeline("text-generation", model="skt/ko-gpt-trinity-1.2B-v0.5")
generator(
        "이 학습 과정에서, 제가 가르치고 싶은 것은",
        max_length=30,
        num_return_sequences=1)

In [ ]:
generator(
        "스스로 학습 목표를 설정하고 학습 계획을 수립하고",
        max_length=30,
        num_return_sequences=1)

In [ ]:
generator(
        "학습자의 학습 의욕을 고취시키는 데 목적이 ",
        max_length=30,
        num_return_sequences=1)